# Mini-projet — modèle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/MiniProjet_modele.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 10 · KOUADIO Kouao Laurent

Binôme : *Nom 1, Nom 2* · Sujet n° : *…*

Ce modèle contient un **exemple complet** (changements eau/terre à
Grand-Lahou). Gardez les sections, remplacez le contenu par votre
sujet. Le notebook doit s'exécuter de haut en bas sans erreur
(*Kernel → Restart & Run All*) avant le rendu.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client odc-stac

## 1. La question

> *Exemple :* entre la saison sèche 2016-2017 et la saison sèche 2025, quelles
> surfaces sont passées de la terre à l'eau (érosion, migration de
> l'embouchure du Bandama) et inversement, autour de Grand-Lahou ?

Une phrase, un lieu, une période. Dites pourquoi c'est important.

## 2. Environnement et paramètres

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from odc.stac import configure_rio, load
from pystac_client import Client

# Tous les choix du projet sont ici, pas cachés dans le code.
BBOX = [-5.12, 5.08, -4.90, 5.19]       # lon min, lat min, lon max, lat max
CRS = "EPSG:32630"                      # UTM 30N
RESOLUTION = 20                         # m (10 m possible, plus lent)
PERIODES = {2017: "2016-12-01/2017-03-31",
            2025: "2024-12-01/2025-03-31"}
NUAGES_MAX = 30                         # % par scène
SEUIL_EAU = 0.0                         # NDWI > seuil = eau

print("Python", sys.version.split()[0], "| xarray", xr.__version__)
configure_rio(cloud_defaults=True, GDAL_HTTP_MAX_RETRY="10",
              GDAL_HTTP_RETRY_DELAY="2")

## 3. Les données

Citez la source exacte et gardez la liste des scènes : c'est ce qui
rend le résultat vérifiable.

In [ ]:
cat = Client.open("https://earth-search.aws.element84.com/v1")
items = {}
for an, dates in PERIODES.items():
    items[an] = cat.search(collections=["sentinel-2-l2a"], bbox=BBOX,
                           datetime=dates,
                           query={"eo:cloud_cover": {"lt": NUAGES_MAX}}
                           ).item_collection()
scenes = pd.DataFrame([{"annee": an, "id": i.id,
                        "date": i.datetime.date(),
                        "nuages_%": round(i.properties["eo:cloud_cover"])}
                       for an, its in items.items() for i in its])
scenes.to_csv("scenes_utilisees.csv", index=False)
scenes

## 4. Prétraitement

Deux pièges vus en cours : les **nuages** (masque SCL) et le
**décalage de réflectance** des images récentes (séance 4).

In [ ]:
def decalage(item):
    """-0,1 seulement si baseline >= 04.00 et décalage non retiré."""
    p = item.properties
    decale = (p.get("s2:processing_baseline", "00.00") >= "04.00"
              and not p.get("earthsearch:boa_offset_applied", False))
    return -0.1 if decale else 0.0


def ndwi_median(its):
    """NDWI médian de la période, nuages masqués."""
    off = {i.datetime.strftime("%Y-%m-%d"): decalage(i) for i in its}
    ds = load(its, bands=["green", "nir", "scl"], bbox=BBOX, crs=CRS,
              resolution=RESOLUTION, chunks={}, groupby="solar_day",
              fail_on_error=False)
    o = xr.DataArray([off[str(t)[:10]] for t in ds.time.values],
                     dims="time", coords={"time": ds.time})
    # on retire nuages, ombres, cirrus et pixels vides ; on garde
    # les classes 2 et 7 où tombe souvent l'eau turbide de la lagune
    ok = ~ds.scl.isin([0, 1, 3, 8, 9, 10])
    g = ds.green.where(ok) * 1e-4 + o
    n = ds.nir.where(ok) * 1e-4 + o
    return ((g - n) / (g + n)).median("time").compute()


ndwi = {an: ndwi_median(its) for an, its in items.items()}
{an: f"{float(v.notnull().mean()) * 100:.0f} % de pixels valides"
 for an, v in ndwi.items()}

## 5. Analyse

$\mathrm{NDWI} = (\rho_{vert} - \rho_{PIR}) / (\rho_{vert} +
\rho_{PIR})$ ; un pixel est de l'eau si NDWI > seuil.

In [ ]:
def changements(seuil):
    a, b = ndwi[2017] > seuil, ndwi[2025] > seuil
    valide = ndwi[2017].notnull() & ndwi[2025].notnull()
    km2 = (RESOLUTION ** 2) / 1e6
    return {"terre -> eau (km²)": float((~a & b & valide).sum()) * km2,
            "eau -> terre (km²)": float((a & ~b & valide).sum()) * km2}


changements(SEUIL_EAU)

## 6. Validation ou sensibilité

Sans données de terrain, montrez au moins que la conclusion ne dépend
pas d'un seul paramètre.

In [ ]:
sens = pd.DataFrame({s: changements(s) for s in (-0.1, 0.0, 0.1)}).T
sens.index.name = "seuil NDWI"
sens.round(2)

## 7. Carte

Règles de la séance 6 : titre, légende, échelle ou coordonnées,
source.

In [ ]:
a, b = ndwi[2017] > SEUIL_EAU, ndwi[2025] > SEUIL_EAU
valide = ndwi[2017].notnull() & ndwi[2025].notnull()
classe = xr.where(~valide, 0, xr.where(a & b, 2, xr.where(~a & b, 3,
                  xr.where(a & ~b, 4, 1))))
couleurs = ["white", "#E9EDE4", "#9CC3D5", "#C0392B", "#2D6A4F"]
fig, ax = plt.subplots(figsize=(10, 5.5))
classe.plot.imshow(ax=ax, cmap=ListedColormap(couleurs), vmin=0,
                   vmax=4, add_colorbar=False)
ax.set_title("Grand-Lahou : eau et terre, déc.-mars 2017 et 2025")
ax.set_xlabel("x UTM 30N (m)")
ax.set_ylabel("y UTM 30N (m)")
ax.legend(handles=[Patch(color=c, label=t) for t, c in zip(
    ["terre (les deux dates)", "eau (les deux dates)",
     "terre devenue eau", "eau devenue terre"], couleurs[1:])],
    loc="lower left")
fig.text(0.01, 0.01, "Source : Sentinel-2 L2A (ESA/Copernicus), "
         "Earth Search", fontsize=9, color="grey")
plt.show()

## 8. Limites et conclusion

*Exemple :* la limite eau/terre dépend de la marée et des vagues au
moment du passage du satellite ; la médiane réduit cet effet sans
l'annuler. La surface « terre devenue eau » change beaucoup avec le
seuil NDWI (section 6) : l'eau turbide de la lagune a un NDWI proche
de 0. Peu de scènes sans nuages en 2017 (bandes blanches = pas de
donnée). Pour conclure à une érosion, il faudrait des relevés GPS du
trait de côte ou des images à très haute résolution.

Écrivez 5 à 8 lignes : ce que montre le résultat, son incertitude, ce
qu'il faudrait pour aller plus loin.

## 9. Liste de contrôle avant le rendu

- [ ] *Restart & Run All* sans erreur
- [ ] sources et dates citées (section 3), liste des scènes enregistrée
- [ ] paramètres regroupés en section 2
- [ ] validation ou analyse de sensibilité (section 6)
- [ ] carte avec titre, légende, source (section 7)
- [ ] limites écrites (section 8)